In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "README.md").is_file() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "README.md").is_file():
    raise FileNotFoundError("Run this notebook from the project directory or one of its subdirectories.")


# FINAL REPRO — EXP04 iTransformer

이 노트북은 **대회 공식 배포 데이터에서 시작하여 최종 submission까지** 한 번에 재현하기 위한 제출용 실행 파일입니다.

## 필요한 입력 파일
- `train_atmos.csv`
- `train_wave.csv`
- `test_context.parquet`
- `test_index.csv`

## 실행 흐름
1. 공식 train 데이터 merge 및 V2 base preprocessing
2. SAFE V4용 관측 mask 생성
3. causal input fill + causal physics features
4. extended causal physics features
5. EXP04 최종 feature set(`ALL_FEATURES`, 31개) 고정
6. Optuna에서 확정된 best parameters 고정
7. iTransformer를 **scratch부터 학습**
8. 모델 / scaler / metadata 저장
9. test inference
10. `submission_exp04_final.csv` 생성

### 규정 관련
- 외부 데이터 사용 없음
- 외부 API / 인터넷 다운로드 없음
- 사전학습 가중치 사용 없음
- 모든 모델 가중치는 본 노트북에서 배포 데이터만으로 학습
- Optuna 탐색은 재현 시간 절약을 위해 다시 수행하지 않고, 기존 탐색의 최적 파라미터를 고정 사용


In [1]:
# ============================================================
# 0. SETUP / REPRODUCIBILITY
# ============================================================
from pathlib import Path
import gc
import json
import math
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

RUN_START = time.time()
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


DEVICE: cpu


In [2]:
# ============================================================
# 1. FINAL CONFIG
# ============================================================
RAW_ATMOS_PATH = Path(PROJECT_ROOT / "data" / "raw" / "train_atmos.csv")
RAW_WAVE_PATH = Path(PROJECT_ROOT / "data" / "raw" / "train_wave.csv")

MERGED_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train.csv")
FINAL_BASE_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_base_v2.csv")
DATA_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_physics_v4.csv")

TEST_CONTEXT_PATH = Path(PROJECT_ROOT / "data" / "raw" / "test_context.parquet")
TEST_INDEX_PATH = Path(PROJECT_ROOT / "data" / "raw" / "test_index.csv")
SUBMISSION_PATH = Path(PROJECT_ROOT / "submission" / "submission_exp04_final.csv")

EXP_DIR = PROJECT_ROOT / "artifacts" / "experiments" / "final_exp04"
EXP_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "time"
STATION_COL = "station"

STEP_MINUTES = 10
STEPS_PER_HOUR = 6

INPUT_LEN = 289
LEAD_HOURS = [3, 6, 9, 12, 18, 24]
LEAD_STEPS = [h * STEPS_PER_HOUR for h in LEAD_HOURS]
MAX_LEAD = max(LEAD_STEPS)
N_TARGETS = len(LEAD_STEPS)

TRAIN_RATIO = 0.80
FINAL_EPOCHS = 35
FINAL_PATIENCE = 6

# EXP04 Optuna best trial (기존 탐색 결과 고정)
BEST_PARAMS = {
    "d_model": 64,
    "n_heads": 8,
    "e_layers": 4,
    "dropout": 0.3460508306239601,
    "lr": 9.11924776137983e-06,
    "weight_decay": 4.815394773868456e-06,
    "batch_size": 64,
}

for p in [RAW_ATMOS_PATH, RAW_WAVE_PATH, TEST_CONTEXT_PATH, TEST_INDEX_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required input file not found: {p}")

print("BEST_PARAMS:", BEST_PARAMS)


BEST_PARAMS: {'d_model': 64, 'n_heads': 8, 'e_layers': 4, 'dropout': 0.3460508306239601, 'lr': 9.11924776137983e-06, 'weight_decay': 4.815394773868456e-06, 'batch_size': 64}


## 2. V2 base preprocessing

아래 코드는 업로드된 `train_v2.ipynb`의 V2 base preprocessing을 그대로 사용합니다.
공식 `train_atmos.csv`, `train_wave.csv`만을 입력으로 사용합니다.


In [ ]:
# ============================================================
# preprocess_v2.py
#
# V2 BASE PREPROCESSING
#
# INPUT
#   train_atmos.csv
#   train_wave.csv
#
# OUTPUT
#   train.csv
#   train_final_base_v2.csv
#
# 핵심 전략
# ------------------------------------------------------------
# 1. 기존 raw merge 구조 유지
# 2. 명백한 물리 범위 이상치만 제거
# 3. 범용 interpolate(limit=2) 제거
# 4. 방향값 일반 linear interpolation 제거
# 5. station별 Hs 전략
#
#    G-ORS:
#       <=3h linear
#       >3h NaN 유지
#
#    I-ORS:
#       <=3h linear
#       >24h G-ORS lag transfer
#
#    S-ORS:
#       <=3h linear
#       >3h NaN 유지
#
# 6. Tp <=3h linear
# 7. Hmax <- station별 Hmax/Hs robust ratio
# 8. Atmos <=1h linear
# 9. wdir/wvdir isolated 1-step circular
# 10. original/imputation flag 보존
# 11. physics feature 생성 X
# ============================================================


from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression


# ============================================================
# PATH
# ============================================================

RAW_ATMOS_PATH = Path(PROJECT_ROOT / "data" / "raw" / "train_atmos.csv")
RAW_WAVE_PATH = Path(PROJECT_ROOT / "data" / "raw" / "train_wave.csv")

MERGED_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train.csv")

FINAL_BASE_PATH = Path(
    PROJECT_ROOT / "data" / "processed" / "train_final_base_v2.csv"
)


# ============================================================
# CONFIG
# ============================================================

TIME_COL = "time"
STATION_COL = "station"

STEP_MINUTES = 10
STEPS_PER_HOUR = 6


BASE_COLUMNS = [
    "hs",
    "tp",
    "hmax",
    "wvdir",
    "wspd",
    "gust",
    "wdir",
    "airt",
    "relh",
    "caph",
]


CONTINUOUS_COLUMNS = [
    "hs",
    "tp",
    "hmax",
    "wspd",
    "gust",
    "airt",
    "relh",
    "caph",
]


DIRECTION_COLUMNS = [
    "wvdir",
    "wdir",
]


# ------------------------------------------------------------
# Hs
# ------------------------------------------------------------

HS_SHORT_MAX_STEPS = 18      # 3h

I_LONG_MIN_STEPS = 145       # >24h

G_TO_I_LAG_STEPS = 17        # 2h50m


# ------------------------------------------------------------
# Tp
# ------------------------------------------------------------

TP_SHORT_MAX_STEPS = 18      # 3h


# ------------------------------------------------------------
# Atmos
# ------------------------------------------------------------

ATMOS_SHORT_MAX_STEPS = 6    # 1h


# ============================================================
# 1. RAW MERGE
# ============================================================

def create_merged_train():

    atmos = pd.read_csv(
        RAW_ATMOS_PATH
    )

    wave = pd.read_csv(
        RAW_WAVE_PATH
    )


    atmos[TIME_COL] = pd.to_datetime(
        atmos[TIME_COL]
    )

    wave[TIME_COL] = pd.to_datetime(
        wave[TIME_COL]
    )


    train = pd.merge(
        atmos,
        wave,
        on=[
            STATION_COL,
            TIME_COL,
        ],
        how="outer",
    )


    train = (
        train
        .sort_values(
            [
                STATION_COL,
                TIME_COL,
            ]
        )
        .reset_index(drop=True)
    )


    preferred_order = [
        "case_id",
        STATION_COL,
        "step_minute",
        TIME_COL,
        "hs",
        "tp",
        "hmax",
        "wvdir",
        "wspd",
        "gust",
        "wdir",
        "airt",
        "relh",
        "caph",
    ]


    existing = [
        c
        for c in preferred_order
        if c in train.columns
    ]

    remaining = [
        c
        for c in train.columns
        if c not in existing
    ]


    train = train[
        existing + remaining
    ]


    train.to_csv(
        MERGED_PATH,
        index=False
    )


    print(
        "Merged:",
        train.shape
    )


    return train


# ============================================================
# 2. ORIGINAL FLAGS
#
# 반드시 어떠한 cleaning보다 먼저 생성
# ============================================================

def add_original_flags(df):

    df = df.copy()


    for col in BASE_COLUMNS:

        df[
            f"{col}_original_observed"
        ] = (
            df[col]
            .notna()
            .astype(np.int8)
        )


        df[
            f"{col}_imputed"
        ] = np.int8(0)


    # Hs는 출처를 상세 기록
    df["hs_fill_method"] = np.where(
        df["hs"].notna(),
        "observed",
        "missing",
    )


    return df


# ============================================================
# 3. CONSERVATIVE ANOMALY CLEANING
#
# 기존 V1의 공격적 규칙은 제거
#
# 삭제한 것:
# - frozen 3point
# - hs diff > 1m
# - 모든 변수 interpolate(limit=2)
# ============================================================

def conservative_anomaly_cleaning(
    df
):

    df = df.copy()


    # --------------------------------------------------------
    # Wave
    # --------------------------------------------------------

    df.loc[
        df["hs"] <= 0,
        "hs"
    ] = np.nan


    df.loc[
        df["tp"] <= 0,
        "tp"
    ] = np.nan


    df.loc[
        df["hmax"] <= 0,
        "hmax"
    ] = np.nan


    # --------------------------------------------------------
    # Wind
    # --------------------------------------------------------

    df.loc[
        df["wspd"] < 0,
        "wspd"
    ] = np.nan


    df.loc[
        df["gust"] < 0,
        "gust"
    ] = np.nan


    # --------------------------------------------------------
    # Atmos
    # --------------------------------------------------------

    df.loc[
        (
            df["caph"] < 950
        )
        |
        (
            df["caph"] > 1050
        ),
        "caph"
    ] = np.nan


    df.loc[
        (
            df["relh"] < 0
        )
        |
        (
            df["relh"] > 100
        ),
        "relh"
    ] = np.nan


    # --------------------------------------------------------
    # Direction normalization
    # --------------------------------------------------------

    for col in DIRECTION_COLUMNS:

        mask = (
            df[col]
            .notna()
        )

        df.loc[
            mask,
            col
        ] = (
            df.loc[
                mask,
                col
            ]
            %
            360
        )


    # --------------------------------------------------------
    # Hmax < Hs
    #
    # 강제로 max() 하지 않고
    # Hmax를 invalid 처리
    # --------------------------------------------------------

    invalid_hmax = (
        df["hmax"].notna()
        &
        df["hs"].notna()
        &
        (
            df["hmax"]
            <
            df["hs"]
        )
    )


    df.loc[
        invalid_hmax,
        "hmax"
    ] = np.nan


    # --------------------------------------------------------
    # gust < wspd
    #
    # 강제로 np.maximum() 하지 않음
    # gust 측 값을 invalid 처리
    # --------------------------------------------------------

    invalid_gust = (
        df["gust"].notna()
        &
        df["wspd"].notna()
        &
        (
            df["gust"]
            <
            df["wspd"]
        )
    )


    df.loc[
        invalid_gust,
        "gust"
    ] = np.nan


    print(
        "\n===== ANOMALY CLEANING ====="
    )

    print(
        "Hmax < Hs removed:",
        int(
            invalid_hmax.sum()
        )
    )

    print(
        "Gust < Wspd removed:",
        int(
            invalid_gust.sum()
        )
    )


    return df


# ============================================================
# 4. GAP UTILITY
# ============================================================

def get_missing_runs(
    series
):

    values = (
        series
        .to_numpy()
    )


    missing = pd.isna(
        values
    )


    runs = []

    pos = 0
    n = len(values)


    while pos < n:

        if not missing[pos]:

            pos += 1
            continue


        start = pos


        while (
            pos < n
            and
            missing[pos]
        ):

            pos += 1


        end = pos - 1


        runs.append(
            (
                start,
                end,
                end - start + 1,
            )
        )


    return runs


# ============================================================
# 5. GENERIC SHORT LINEAR INTERPOLATION
#
# gap 전체 길이가 threshold 이하일 때만
# 양쪽 boundary가 존재해야 함
# ============================================================

def fill_short_linear(
    df,
    col,
    max_steps,
    stations=None,
    method_name=None,
):

    total = 0


    if stations is None:

        stations = (
            df[STATION_COL]
            .dropna()
            .unique()
        )


    flag_col = (
        f"{col}_imputed"
    )


    for station in stations:

        index = df.index[
            df[STATION_COL]
            ==
            station
        ]


        s = (
            df.loc[
                index,
                col
            ]
            .copy()
        )


        runs = (
            get_missing_runs(
                s
            )
        )


        for (
            start,
            end,
            gap_len
        ) in runs:


            if (
                gap_len
                >
                max_steps
            ):

                continue


            left = (
                start - 1
            )

            right = (
                end + 1
            )


            if (
                left < 0
                or
                right >= len(s)
            ):

                continue


            left_value = (
                s.iloc[
                    left
                ]
            )

            right_value = (
                s.iloc[
                    right
                ]
            )


            if (
                pd.isna(
                    left_value
                )
                or
                pd.isna(
                    right_value
                )
            ):

                continue


            fill_values = np.linspace(
                left_value,
                right_value,
                gap_len + 2,
            )[1:-1]


            fill_index = (
                index[
                    start:
                    end + 1
                ]
            )


            df.loc[
                fill_index,
                col
            ] = fill_values


            df.loc[
                fill_index,
                flag_col
            ] = 1


            if (
                col == "hs"
                and
                method_name is not None
            ):

                df.loc[
                    fill_index,
                    "hs_fill_method"
                ] = method_name


            # local copy 갱신
            s.iloc[
                start:
                end + 1
            ] = fill_values


            total += gap_len


    return total


# ============================================================
# 6. EXACT 1-STEP LINEAR
#
# 모든 continuous variable 공통
# ============================================================

def fill_exact_single_linear(
    df
):

    print(
        "\n===== EXACT 1-STEP LINEAR ====="
    )


    summary = {}


    for col in CONTINUOUS_COLUMNS:

        method = (
            "single_linear"
            if col == "hs"
            else None
        )


        n = fill_short_linear(
            df=df,
            col=col,
            max_steps=1,
            method_name=method,
        )


        summary[col] = n


    print(
        pd.Series(
            summary,
            name="filled"
        )
    )


    return df


# ============================================================
# 7. CIRCULAR INTERPOLATION
#
# exact isolated 1-step only
# ============================================================

def circular_midpoint(
    a,
    b
):

    a = np.deg2rad(a)
    b = np.deg2rad(b)


    x = (
        np.cos(a)
        +
        np.cos(b)
    )

    y = (
        np.sin(a)
        +
        np.sin(b)
    )


    return (
        np.rad2deg(
            np.arctan2(
                y,
                x
            )
        )
        %
        360
    )


def fill_single_direction(
    df,
    col
):

    total = 0


    for station in (
        df[STATION_COL]
        .dropna()
        .unique()
    ):

        index = df.index[
            df[STATION_COL]
            ==
            station
        ]


        values = (
            df.loc[
                index,
                col
            ]
            .to_numpy(
                dtype=float
            )
        )


        original_missing = (
            np.isnan(
                values
            )
        )


        for i in range(
            1,
            len(values) - 1
        ):

            # 반드시 원래 isolated 1-step
            if not original_missing[i]:

                continue


            if (
                original_missing[
                    i - 1
                ]
                or
                original_missing[
                    i + 1
                ]
            ):

                continue


            if (
                not np.isfinite(
                    values[
                        i - 1
                    ]
                )
                or
                not np.isfinite(
                    values[
                        i + 1
                    ]
                )
            ):

                continue


            value = (
                circular_midpoint(
                    values[
                        i - 1
                    ],
                    values[
                        i + 1
                    ],
                )
            )


            global_index = (
                index[i]
            )


            df.loc[
                global_index,
                col
            ] = value


            df.loc[
                global_index,
                f"{col}_imputed"
            ] = 1


            total += 1


    return total


def fill_direction_gaps(
    df
):

    print(
        "\n===== DIRECTION 1-STEP ====="
    )


    for col in DIRECTION_COLUMNS:

        n = fill_single_direction(
            df,
            col
        )

        print(
            col,
            ":",
            n
        )


    return df


# ============================================================
# 8. STATION-SPECIFIC HS SHORT GAPS
#
# exact 1-step은 앞 단계에서 이미 채워짐
#
# 여기서는 남은 gap 중 <=3h
# ============================================================

def fill_station_hs_short_gaps(
    df
):

    print(
        "\n===== STATION-SPECIFIC HS ====="
    )


    # --------------------------------------------------------
    # G
    # --------------------------------------------------------

    g_n = fill_short_linear(
        df=df,
        col="hs",
        max_steps=HS_SHORT_MAX_STEPS,
        stations=["G-ORS"],
        method_name="short_linear",
    )


    # --------------------------------------------------------
    # I
    # --------------------------------------------------------

    i_n = fill_short_linear(
        df=df,
        col="hs",
        max_steps=HS_SHORT_MAX_STEPS,
        stations=["I-ORS"],
        method_name="short_linear",
    )


    # --------------------------------------------------------
    # S
    # --------------------------------------------------------

    s_n = fill_short_linear(
        df=df,
        col="hs",
        max_steps=HS_SHORT_MAX_STEPS,
        stations=["S-ORS"],
        method_name="short_linear",
    )


    print(
        "G-ORS short Hs:",
        g_n
    )

    print(
        "I-ORS short Hs:",
        i_n
    )

    print(
        "S-ORS short Hs:",
        s_n
    )


    return df


# ============================================================
# 9. I-ORS LONG GAP
#
# I(t) <- a * G(t - 17step) + b
#
# calibration:
# - gap 시작 이전만
# - original observed I 사용
# - original observed G 사용
#
# source prediction:
# - G short interpolation 값은 사용 가능
#   (과거 input reconstruction 목적)
# ============================================================

def fill_i_long_gap_from_g(
    df
):

    print(
        "\n===== I-ORS LONG GAP <- G-ORS ====="
    )


    # --------------------------------------------------------
    # 현재 Hs wide
    # --------------------------------------------------------

    hs_wide = (
        df
        .pivot(
            index=TIME_COL,
            columns=STATION_COL,
            values="hs",
        )
        .sort_index()
    )


    g_lag = (
        hs_wide[
            "G-ORS"
        ]
        .shift(
            G_TO_I_LAG_STEPS
        )
    )


    # --------------------------------------------------------
    # original observation wide
    # --------------------------------------------------------

    obs_wide = (
        df
        .pivot(
            index=TIME_COL,
            columns=STATION_COL,
            values="hs_original_observed",
        )
        .sort_index()
    )


    g_obs_lag = (
        obs_wide[
            "G-ORS"
        ]
        .shift(
            G_TO_I_LAG_STEPS
        )
    )


    i_mask = (
        df[STATION_COL]
        ==
        "I-ORS"
    )


    i_df = (
        df.loc[
            i_mask,
            [
                TIME_COL,
                "hs",
            ]
        ]
        .sort_values(
            TIME_COL
        )
        .copy()
    )


    i_indices = (
        i_df.index
        .to_numpy()
    )


    runs = get_missing_runs(
        i_df["hs"]
    )


    long_runs = [
        run
        for run in runs
        if run[2]
        >=
        I_LONG_MIN_STEPS
    ]


    print(
        "Long I gaps:",
        len(
            long_runs
        )
    )


    total_fill = 0


    for number, (
        start,
        end,
        gap_len
    ) in enumerate(
        long_runs,
        1
    ):


        gap_start = (
            i_df.iloc[
                start
            ][
                TIME_COL
            ]
        )


        gap_end = (
            i_df.iloc[
                end
            ][
                TIME_COL
            ]
        )


        # ====================================================
        # CALIBRATION
        #
        # 반드시 gap 이전만
        # ====================================================

        calibration = (
            pd.DataFrame({

                "I":
                    hs_wide[
                        "I-ORS"
                    ],

                "G_lag":
                    g_lag,

                "I_obs":
                    obs_wide[
                        "I-ORS"
                    ],

                "G_obs_lag":
                    g_obs_lag,
            })
        )


        calibration = calibration[
            calibration.index
            <
            gap_start
        ]


        calibration = calibration[
            (
                calibration[
                    "I_obs"
                ]
                ==
                1
            )
            &
            (
                calibration[
                    "G_obs_lag"
                ]
                ==
                1
            )
        ]


        calibration = (
            calibration[
                [
                    "I",
                    "G_lag",
                ]
            ]
            .dropna()
        )


        print(
            f"\nGap {number}:",
            gap_start,
            "->",
            gap_end,
        )


        print(
            "steps:",
            gap_len,
            "| days:",
            round(
                gap_len
                /
                144,
                2
            )
        )


        print(
            "calibration:",
            len(
                calibration
            )
        )


        if (
            len(
                calibration
            )
            <
            1000
        ):

            print(
                "SKIP - calibration 부족"
            )

            continue


        model = LinearRegression()


        model.fit(
            calibration[
                [
                    "G_lag"
                ]
            ],
            calibration[
                "I"
            ],
        )


        print(
            "coef:",
            round(
                model.coef_[0],
                5
            )
        )


        print(
            "intercept:",
            round(
                model.intercept_,
                5
            )
        )


        # ====================================================
        # PREDICTION
        # ====================================================

        gap_times = (
            i_df.iloc[
                start:
                end + 1
            ][
                TIME_COL
            ]
        )


        source = (
            g_lag
            .reindex(
                gap_times
            )
        )


        valid_source = (
            source
            .notna()
            .to_numpy()
        )


        print(
            "source coverage:",
            f"{valid_source.mean() * 100:.2f}%"
        )


        if (
            valid_source.sum()
            ==
            0
        ):

            continue


        pred = np.full(
            gap_len,
            np.nan,
            dtype=float,
        )


        pred[
            valid_source
        ] = (
            model.predict(
                source[
                    valid_source
                ]
                .to_numpy()
                .reshape(
                    -1,
                    1
                )
            )
        )


        pred = np.where(
            np.isfinite(
                pred
            ),
            np.clip(
                pred,
                0.01,
                None
            ),
            np.nan,
        )


        global_index = (
            i_indices[
                start:
                end + 1
            ]
        )


        currently_missing = (
            df.loc[
                global_index,
                "hs"
            ]
            .isna()
            .to_numpy()
        )


        fillable = (
            valid_source
            &
            currently_missing
            &
            np.isfinite(
                pred
            )
        )


        fill_index = (
            global_index[
                fillable
            ]
        )


        df.loc[
            fill_index,
            "hs"
        ] = (
            pred[
                fillable
            ]
        )


        df.loc[
            fill_index,
            "hs_imputed"
        ] = 1


        df.loc[
            fill_index,
            "hs_fill_method"
        ] = (
            "G_lag_transfer"
        )


        total_fill += len(
            fill_index
        )


    print(
        "\nI transfer total:",
        total_fill
    )


    return df


# ============================================================
# 10. TP SHORT GAP
#
# <=3h
# ============================================================

def fill_tp_short_gaps(
    df
):

    print(
        "\n===== TP <=3H ====="
    )


    n = fill_short_linear(
        df=df,
        col="tp",
        max_steps=TP_SHORT_MAX_STEPS,
    )


    print(
        "Tp filled:",
        n
    )


    return df


# ============================================================
# 11. HMAX RECONSTRUCTION
#
# station별 Hmax/Hs median
#
# ratio calibration에는
# original observed pair만 사용
# ============================================================

def reconstruct_hmax(
    df
):

    print(
        "\n===== HMAX <- HS ====="
    )


    calibration = df[
        (
            df[
                "hs_original_observed"
            ]
            ==
            1
        )
        &
        (
            df[
                "hmax_original_observed"
            ]
            ==
            1
        )
        &
        df["hs"].notna()
        &
        df["hmax"].notna()
        &
        (
            df["hs"]
            >
            0
        )
    ].copy()


    calibration[
        "_ratio"
    ] = (
        calibration[
            "hmax"
        ]
        /
        calibration[
            "hs"
        ]
    )


    # 이상 ratio 제외
    calibration = calibration[
        calibration[
            "_ratio"
        ]
        .between(
            1.0,
            3.0
        )
    ]


    station_ratio = (
        calibration
        .groupby(
            STATION_COL
        )[
            "_ratio"
        ]
        .median()
    )


    print(
        "\nHmax/Hs ratio:"
    )

    print(
        station_ratio
    )


    total = 0


    for station, ratio in (
        station_ratio.items()
    ):


        mask = (
            (
                df[STATION_COL]
                ==
                station
            )
            &
            df["hmax"].isna()
            &
            df["hs"].notna()
        )


        n = int(
            mask.sum()
        )


        df.loc[
            mask,
            "hmax"
        ] = (
            df.loc[
                mask,
                "hs"
            ]
            *
            ratio
        )


        df.loc[
            mask,
            "hmax_imputed"
        ] = 1


        total += n


    print(
        "\nHmax filled:",
        total
    )


    return df


# ============================================================
# 12. ATMOS SHORT GAP
#
# <=1h
#
# wdir는 circular에서 이미 처리했으므로 제외
# ============================================================

def fill_short_atmos(
    df
):

    print(
        "\n===== ATMOS <=1H ====="
    )


    atmos_columns = [
        "wspd",
        "gust",
        "airt",
        "relh",
        "caph",
    ]


    summary = {}


    for col in (
        atmos_columns
    ):

        n = fill_short_linear(
            df=df,
            col=col,
            max_steps=ATMOS_SHORT_MAX_STEPS,
        )


        summary[col] = n


    print(
        pd.Series(
            summary,
            name="filled"
        )
    )


    return df


# ============================================================
# 13. FINAL SANITY CHECK
# ============================================================

def final_sanity(
    df
):

    df = df.copy()


    # --------------------------------------------------------
    # Base physical bounds
    # --------------------------------------------------------

    for col in [
        "hs",
        "tp",
        "hmax",
    ]:

        df.loc[
            df[col]
            <=
            0,
            col
        ] = np.nan


    df.loc[
        df["wspd"]
        <
        0,
        "wspd"
    ] = np.nan


    df.loc[
        df["gust"]
        <
        0,
        "gust"
    ] = np.nan


    df.loc[
        ~df["relh"].between(
            0,
            100,
        ),
        "relh"
    ] = np.nan


    df.loc[
        ~df["caph"].between(
            950,
            1050,
        ),
        "caph"
    ] = np.nan


    for col in DIRECTION_COLUMNS:

        df[col] = (
            df[col]
            %
            360
        )


    # --------------------------------------------------------
    # hmax physical relation
    # --------------------------------------------------------

    bad_hmax = (
        df["hmax"].notna()
        &
        df["hs"].notna()
        &
        (
            df["hmax"]
            <
            df["hs"]
        )
    )


    print(
        "\nHmax < Hs violations:",
        int(
            bad_hmax.sum()
        )
    )


    return df


# ============================================================
# 14. DIAGNOSTICS
# ============================================================

def print_diagnostics(
    df
):

    print(
        "\n"
        +
        "=" * 80
    )

    print(
        "FINAL V2 DIAGNOSTICS"
    )

    print(
        "=" * 80
    )


    # --------------------------------------------------------
    # Missing
    # --------------------------------------------------------

    missing = pd.DataFrame({

        "missing":
            df[
                BASE_COLUMNS
            ]
            .isna()
            .sum(),

        "missing_pct":
            (
                df[
                    BASE_COLUMNS
                ]
                .isna()
                .mean()
                *
                100
            ),
    })


    print(
        "\n===== FINAL MISSING ====="
    )

    print(
        missing
        .sort_values(
            "missing_pct",
            ascending=False
        )
        .round(3)
    )


    # --------------------------------------------------------
    # Hs by station
    # --------------------------------------------------------

    print(
        "\n===== HS BY STATION ====="
    )


    hs_station = (
        df
        .groupby(
            STATION_COL
        )
        .agg(

            rows=(
                "hs",
                "size"
            ),

            original_hs=(
                "hs_original_observed",
                "sum"
            ),

            hs_available=(
                "hs",
                lambda s:
                s.notna().sum()
            ),

            hs_missing=(
                "hs",
                lambda s:
                s.isna().sum()
            ),

            hs_imputed=(
                "hs_imputed",
                "sum"
            ),
        )
    )


    print(
        hs_station
    )


    # --------------------------------------------------------
    # Hs source
    # --------------------------------------------------------

    print(
        "\n===== HS FILL METHOD ====="
    )


    print(
        pd.crosstab(
            df[
                STATION_COL
            ],
            df[
                "hs_fill_method"
            ],
        )
    )


    # --------------------------------------------------------
    # Imputation
    # --------------------------------------------------------

    print(
        "\n===== IMPUTED COUNTS ====="
    )


    flag_cols = [
        c
        for c in df.columns
        if c.endswith(
            "_imputed"
        )
    ]


    print(
        df[
            flag_cols
        ]
        .sum()
        .sort_values(
            ascending=False
        )
    )


    # --------------------------------------------------------
    # Remaining long Hs gaps
    # --------------------------------------------------------

    print(
        "\n===== REMAINING HS LONG GAPS ====="
    )


    records = []


    for station, group in (
        df.groupby(
            STATION_COL
        )
    ):

        group = (
            group
            .sort_values(
                TIME_COL
            )
        )


        runs = get_missing_runs(
            group["hs"]
        )


        for (
            start,
            end,
            n
        ) in runs:

            if n > 18:

                records.append({

                    "station":
                        station,

                    "start":
                        group.iloc[
                            start
                        ][
                            TIME_COL
                        ],

                    "end":
                        group.iloc[
                            end
                        ][
                            TIME_COL
                        ],

                    "steps":
                        n,

                    "hours":
                        n
                        /
                        6,

                    "days":
                        n
                        /
                        144,
                })


    long_gaps = pd.DataFrame(
        records
    )


    if len(
        long_gaps
    ):

        print(
            long_gaps
            .sort_values(
                "steps",
                ascending=False
            )
            .head(
                30
            )
        )

    else:

        print(
            "No >3h Hs gaps"
        )


# ============================================================
# 15. COLUMN ORDER
# ============================================================

def arrange_columns(
    df
):

    identity = [
        c
        for c in [
            "case_id",
            STATION_COL,
            "step_minute",
            TIME_COL,
        ]
        if c in df.columns
    ]


    base = [
        c
        for c in BASE_COLUMNS
        if c in df.columns
    ]


    observed = [
        f"{c}_original_observed"
        for c in BASE_COLUMNS
        if (
            f"{c}_original_observed"
            in df.columns
        )
    ]


    imputed = [
        f"{c}_imputed"
        for c in BASE_COLUMNS
        if (
            f"{c}_imputed"
            in df.columns
        )
    ]


    extra = [
        "hs_fill_method"
    ]


    ordered = (
        identity
        +
        base
        +
        observed
        +
        imputed
        +
        extra
    )


    remaining = [
        c
        for c in df.columns
        if c not in ordered
    ]


    return df[
        ordered
        +
        remaining
    ]


# ============================================================
# 16. MAIN
# ============================================================

def main():

    # --------------------------------------------------------
    # RAW
    # --------------------------------------------------------

    df = create_merged_train()


    print(
        "\nInitial shape:",
        df.shape
    )


    # --------------------------------------------------------
    # 가장 먼저 원본 여부 저장
    # --------------------------------------------------------

    df = add_original_flags(
        df
    )


    # --------------------------------------------------------
    # conservative anomaly
    # --------------------------------------------------------

    df = conservative_anomaly_cleaning(
        df
    )


    # --------------------------------------------------------
    # exact 1-step continuous
    # --------------------------------------------------------

    df = fill_exact_single_linear(
        df
    )


    # --------------------------------------------------------
    # direction isolated 1-step
    # --------------------------------------------------------

    df = fill_direction_gaps(
        df
    )


    # --------------------------------------------------------
    # station-specific Hs
    # --------------------------------------------------------

    df = fill_station_hs_short_gaps(
        df
    )


    # --------------------------------------------------------
    # I long outage <- G
    # --------------------------------------------------------

    df = fill_i_long_gap_from_g(
        df
    )


    # --------------------------------------------------------
    # Tp <=3h
    # --------------------------------------------------------

    df = fill_tp_short_gaps(
        df
    )


    # --------------------------------------------------------
    # Hmax reconstruction
    # --------------------------------------------------------

    df = reconstruct_hmax(
        df
    )


    # --------------------------------------------------------
    # Atmos <=1h
    # --------------------------------------------------------

    df = fill_short_atmos(
        df
    )


    # --------------------------------------------------------
    # sanity
    # --------------------------------------------------------

    df = final_sanity(
        df
    )


    # --------------------------------------------------------
    # sort
    # --------------------------------------------------------

    df = (
        df
        .sort_values(
            [
                STATION_COL,
                TIME_COL,
            ]
        )
        .reset_index(
            drop=True
        )
    )


    df = arrange_columns(
        df
    )


    # --------------------------------------------------------
    # diagnostics
    # --------------------------------------------------------

    print_diagnostics(
        df
    )


    # --------------------------------------------------------
    # SAVE ONCE
    # --------------------------------------------------------

    df.to_csv(
        FINAL_BASE_PATH,
        index=False
    )


    print(
        "\n"
        +
        "=" * 80
    )

    print(
        "SAVED"
    )

    print(
        "=" * 80
    )


    print(
        FINAL_BASE_PATH
    )


    print(
        "shape:",
        df.shape
    )


    print(
        "\nPhysics features included:",
        False
    )


    return df


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    final_v2 = main()

Merged: (183600, 12)

Initial shape: (183600, 12)

===== ANOMALY CLEANING =====
Hmax < Hs removed: 0
Gust < Wspd removed: 5561

===== EXACT 1-STEP LINEAR =====


In [ ]:
# ============================================================
# 3. BUILD SAFE-V4 MASKS FROM V2 ORIGINAL-OBSERVATION FLAGS
# ============================================================
# train_v2.ipynb는 별도의 synthetic grid row를 생성하지 않으므로
# grid_inserted는 0으로 둔다.
# wave_available / atmos_available / hs_observed는
# V2가 cleaning/imputation 전에 저장한 *_original_observed flag에서 생성한다.

base = pd.read_csv(FINAL_BASE_PATH, parse_dates=[TIME_COL])

required_original_flags = [
    "hs_original_observed", "tp_original_observed",
    "hmax_original_observed", "wvdir_original_observed",
    "wspd_original_observed", "gust_original_observed",
    "wdir_original_observed", "airt_original_observed",
    "relh_original_observed", "caph_original_observed",
]
missing = [c for c in required_original_flags if c not in base.columns]
if missing:
    raise ValueError(f"V2 output is missing original-observation flags: {missing}")

base["grid_inserted"] = np.int8(0)

base["wave_available"] = (
    base[[
        "hs_original_observed",
        "tp_original_observed",
        "hmax_original_observed",
        "wvdir_original_observed",
    ]].max(axis=1)
).astype("int8")

base["atmos_available"] = (
    base[[
        "wspd_original_observed",
        "gust_original_observed",
        "wdir_original_observed",
        "airt_original_observed",
        "relh_original_observed",
        "caph_original_observed",
    ]].max(axis=1)
).astype("int8")

base["hs_observed"] = base["hs_original_observed"].astype("int8")

# SAFE V4 input 파일명으로 명시적 저장
SAFE_V4_INPUT_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_v2.csv")
base.to_csv(SAFE_V4_INPUT_PATH, index=False)

print("SAFE V4 input:", SAFE_V4_INPUT_PATH, base.shape)
print(base[["grid_inserted", "wave_available", "atmos_available", "hs_observed"]].sum())


## 4. SAFE V4 — authoritative causal preprocessing

원본 `train_v4(1).ipynb`에서 RAW 셀이었던 **SAFE V4 FINAL BUILD**를
제출본에서는 실행 가능한 code cell로 사용합니다.

이전 V3-style `bfill/ffill` 전처리(Cell 2~6)는 포함하지 않습니다.


In [ ]:
# ============================================================
# 4. SAFE V4 FINAL BUILD: CAUSAL INPUT FILL, OBSERVED TARGETS
# ============================================================
# This cell is the authoritative V4 output step. It intentionally
# rereads train_v2.csv so the earlier V3-style imputation cells do
# not affect train_final_physics_v4.csv.

import numpy as np
import pandas as pd

INPUT_PATH = str(SAFE_V4_INPUT_PATH)
OUTPUT_PATH = str(DATA_PATH)
VALUE_COLUMNS = [
    "hs", "tp", "hmax", "wvdir", "wspd",
    "gust", "wdir", "airt", "relh", "caph",
]
WAVE_COLUMNS = ["hs", "tp", "hmax", "wvdir"]
ATMOS_COLUMNS = ["wspd", "gust", "wdir", "airt", "relh", "caph"]
SHORT_GAP_STEPS = 1  # Only a single preceding 10-minute value may be carried forward.

raw = pd.read_csv(INPUT_PATH, parse_dates=["time"])
required_masks = ["grid_inserted", "wave_available", "atmos_available", "hs_observed"]
missing_masks = [col for col in required_masks if col not in raw.columns]
if missing_masks:
    raise ValueError(f"train_v2.csv is missing required masks: {missing_masks}")

def prepare_station_causally(group):
    g = group.sort_values("time").copy()

    # Preserve the raw hs label mask for every later training/OOF split.
    g["hs_original_observed"] = g["hs_observed"].astype("int8")

    # Directions are circular values; normalizing does not impute them.
    for col in ["wdir", "wvdir"]:
        g.loc[g[col].notna(), col] = g.loc[g[col].notna(), col] % 360.0

    values_before_fill = g[VALUE_COLUMNS].copy()
    causal_values = values_before_fill.ffill(limit=SHORT_GAP_STEPS)
    imputed = values_before_fill.isna() & causal_values.notna()

    # A row inserted for a complete source outage must remain unusable.
    inserted = g["grid_inserted"].eq(1)
    causal_values.loc[inserted, VALUE_COLUMNS] = np.nan
    imputed.loc[inserted, VALUE_COLUMNS] = False
    g[VALUE_COLUMNS] = causal_values

    for col in VALUE_COLUMNS:
        g[f"{col}_input_imputed"] = imputed[col].astype("int8")

    g["input_complete"] = (
        (~inserted) & g[VALUE_COLUMNS].notna().all(axis=1)
    ).astype("int8")
    return g

prepared = pd.concat(
    [prepare_station_causally(group) for _, group in raw.groupby("station", sort=True)],
    ignore_index=True,
)
prepared = prepared.sort_values(["station", "time"]).reset_index(drop=True)

def add_physics_features(group):
    g = group.sort_values("time").copy()

    # All rolling windows require a complete causal history. No bfill/ffill follows.
    g["wspd_mean_6h"] = g["wspd"].rolling(36, min_periods=36).mean()
    g["wspd_mean_12h"] = g["wspd"].rolling(72, min_periods=72).mean()
    g["gust_max_6h"] = g["gust"].rolling(36, min_periods=36).max()
    g["gust_max_12h"] = g["gust"].rolling(72, min_periods=72).max()
    g["gust_minus_wspd"] = g["gust"] - g["wspd"]

    direction_diff = (g["wdir"] - g["wvdir"] + 180.0) % 360.0 - 180.0
    g["wind_wave_diff"] = np.abs(direction_diff)
    g["wind_wave_alignment"] = np.cos(np.deg2rad(direction_diff))

    g["caph_change_3h"] = g["caph"] - g["caph"].shift(18)
    g["caph_change_6h"] = g["caph"] - g["caph"].shift(36)
    g["caph_change_12h"] = g["caph"] - g["caph"].shift(72)

    g["hs_diff_1h"] = g["hs"] - g["hs"].shift(6)
    g["hs_diff_3h"] = g["hs"] - g["hs"].shift(18)
    g["hs_mean_6h"] = g["hs"].rolling(36, min_periods=36).mean()
    g["hs_mean_12h"] = g["hs"].rolling(72, min_periods=72).mean()
    g["hs_max_6h"] = g["hs"].rolling(36, min_periods=36).max()
    g["hs_max_12h"] = g["hs"].rolling(72, min_periods=72).max()

    wavelength = 1.56 * (g["tp"] ** 2)
    g["wave_steepness"] = g["hs"] / np.maximum(wavelength, 1.0)
    g["wave_energy"] = g["hs"] ** 2
    g["effective_wind_forcing"] = (g["wspd"] ** 2) * g["wind_wave_alignment"]

    wind_radians = np.deg2rad(g["wdir"])
    wave_radians = np.deg2rad(g["wvdir"])
    g["u_wind"] = g["wspd"] * np.sin(wind_radians)
    g["v_wind"] = g["wspd"] * np.cos(wind_radians)
    g["u_wave"] = g["hs"] * np.sin(wave_radians)
    g["v_wave"] = g["hs"] * np.cos(wave_radians)
    return g.replace([np.inf, -np.inf], np.nan)

physics = pd.concat(
    [add_physics_features(group) for _, group in prepared.groupby("station", sort=True)],
    ignore_index=True,
)
physics = physics.sort_values(["station", "time"]).reset_index(drop=True)

assert not physics.duplicated(["station", "time"]).any()
assert physics.loc[physics["grid_inserted"].eq(1), VALUE_COLUMNS].isna().all().all()
assert physics.loc[physics["hs_original_observed"].eq(1), "hs"].notna().all()

physics.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

summary = physics.groupby("station", as_index=False).agg(
    rows=("time", "size"),
    grid_inserted=("grid_inserted", "sum"),
    valid_input_rows=("input_complete", "sum"),
    observed_hs_targets=("hs_original_observed", "sum"),
)
print(f"Saved safe V4 dataset: {OUTPUT_PATH} | shape: {physics.shape}")
display(summary)


## 5. Extended causal physics features

In [ ]:
# ============================================================
# 5. EXTENDED CAUSAL PHYSICS FEATURES FOR EXP14
# ============================================================
import numpy as np
import pandas as pd

OUTPUT_PATH = str(DATA_PATH)
physics = pd.read_csv(OUTPUT_PATH, parse_dates=["time"])

def add_extended_features(group):
    g = group.sort_values("time").copy()

    # Wave state and variability. Full windows prevent partial-history leakage.
    g["hmax_hs_ratio"] = g["hmax"] / g["hs"].clip(lower=0.01)
    g["wave_energy_flux"] = (g["hs"] ** 2) * g["tp"]
    g["hs_std_6h"] = g["hs"].rolling(36, min_periods=36).std()
    g["hs_std_12h"] = g["hs"].rolling(72, min_periods=72).std()
    g["hs_range_6h"] = g["hs"].rolling(36, min_periods=36).max() - g["hs"].rolling(36, min_periods=36).min()
    g["hs_range_12h"] = g["hs"].rolling(72, min_periods=72).max() - g["hs"].rolling(72, min_periods=72).min()

    # Wind evolution and the wind component perpendicular to wave travel.
    g["wspd_diff_1h"] = g["wspd"] - g["wspd"].shift(6)
    g["wspd_diff_3h"] = g["wspd"] - g["wspd"].shift(18)
    g["wspd_std_6h"] = g["wspd"].rolling(36, min_periods=36).std()
    g["wspd_std_12h"] = g["wspd"].rolling(72, min_periods=72).std()
    g["wind_cross_wave"] = g["wspd"] * np.sin(np.deg2rad(g["wdir"] - g["wvdir"]))

    # Atmospheric state: Magnus dew point and a trailing 24-hour pressure anomaly.
    rh_fraction = g["relh"].clip(lower=1e-3, upper=100.0) / 100.0
    gamma = np.log(rh_fraction) + (17.625 * g["airt"]) / (243.04 + g["airt"])
    g["dew_point"] = (243.04 * gamma) / (17.625 - gamma)
    g["caph_anomaly_24h"] = g["caph"] - g["caph"].rolling(144, min_periods=144).mean()

    # Deterministic calendar cycles are available at inference time.
    hour = g["time"].dt.hour + g["time"].dt.minute / 60.0
    day_of_year = g["time"].dt.dayofyear
    g["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    g["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)
    g["doy_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
    g["doy_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)
    return g.replace([np.inf, -np.inf], np.nan)

physics = pd.concat(
    [add_extended_features(group) for _, group in physics.groupby("station", sort=True)],
    ignore_index=True,
)
physics = physics.sort_values(["station", "time"]).reset_index(drop=True)
physics.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

new_features = [
    "hmax_hs_ratio", "wave_energy_flux", "hs_std_6h", "hs_std_12h",
    "hs_range_6h", "hs_range_12h", "wspd_diff_1h", "wspd_diff_3h",
    "wspd_std_6h", "wspd_std_12h", "wind_cross_wave", "dew_point",
    "caph_anomaly_24h", "hour_sin", "hour_cos", "doy_sin", "doy_cos",
]
print(f"Extended V4 features saved: {OUTPUT_PATH}")
print(pd.DataFrame({"missing": physics[new_features].isna().sum()}).T)


In [ ]:
# ============================================================
# 6. LOAD FINAL TRAIN FRAME
# ============================================================
df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL])
df = df.sort_values([STATION_COL, TIME_COL]).reset_index(drop=True)

print("train data:", df.shape)
print("stations:", sorted(df[STATION_COL].dropna().unique().tolist()))
print("time:", df[TIME_COL].min(), "~", df[TIME_COL].max())


In [ ]:
# ============================================================
# 7. FIXED EXP04 FEATURE SET — ALL_FEATURES
# ============================================================
BEST_FEATURE_NAME = "ALL_FEATURES"

BEST_FEATURES = [
    "hs", "tp", "hmax",
    "wspd", "gust", "u_wind", "v_wind",
    "hs_diff_1h", "hs_diff_3h",
    "hs_mean_6h", "hs_mean_12h",
    "hs_max_6h", "hs_max_12h",
    "wave_steepness", "wave_energy",
    "effective_wind_forcing", "u_wave", "v_wave",
    "wspd_mean_6h", "wspd_mean_12h",
    "gust_max_6h", "gust_max_12h", "gust_minus_wspd",
    "caph_change_3h", "caph_change_6h", "caph_change_12h",
    "wind_wave_alignment", "wind_wave_diff",
    "airt", "relh", "caph",
]

missing = [c for c in BEST_FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"Missing EXP04 final features: {missing}")

print("FEATURE SET:", BEST_FEATURE_NAME)
print("N FEATURES:", len(BEST_FEATURES))
print(BEST_FEATURES)


## 8. Sample building / chronological split / scaling

In [ ]:
# ============================================================
# 5. SAMPLE BUILDING
# ============================================================
STATION_TO_ID = {station: i for i, station in enumerate(sorted(df[STATION_COL].unique()))}

def build_samples(frame, features, input_len=INPUT_LEN, lead_steps=LEAD_STEPS):
    """Store window metadata only; never materialize an (n_windows, L, F) array."""
    station_id, start_idx, end_idx = [], [], []
    origin_hs, origin_time = [], []
    x_by_station, hs_by_station = {}, {}
    max_lead = max(lead_steps)

    for station, g in frame.groupby(STATION_COL, sort=False):
        g = g.sort_values(TIME_COL).reset_index(drop=True)
        sid = STATION_TO_ID[station]
        # One 2D matrix per station is retained and sliced lazily by WaveDataset.
        Xv = g.loc[:, features].to_numpy(dtype=np.float32, copy=True)
        hsv = g["hs"].to_numpy(dtype=np.float32, copy=True)
        obs = g["hs_original_observed"].to_numpy(dtype=np.int8, copy=True)
        times = g[TIME_COL].to_numpy(copy=True)
        x_by_station[sid] = Xv
        hs_by_station[sid] = hsv
        dt = pd.Series(g[TIME_COL]).diff().dt.total_seconds().div(60).to_numpy()

        for e in range(input_len - 1, len(g) - max_lead):
            s = e - input_len + 1
            if not np.all(dt[s + 1:e + 1] == STEP_MINUTES):
                continue
            # This is a temporary view used only to validate the candidate window.
            if not np.isfinite(Xv[s:e + 1]).all():
                continue
            target_idx = np.asarray([e + step for step in lead_steps], dtype=np.int64)
            if not np.isfinite(hsv[target_idx]).all() or not np.all(obs[target_idx] == 1):
                continue
            if not np.isfinite(hsv[e]):
                continue

            station_id.append(sid)
            start_idx.append(s)
            end_idx.append(e)
            origin_hs.append(hsv[e])
            origin_time.append(times[e])

    return {
        "station_id": np.asarray(station_id, dtype=np.int64),
        "start_idx": np.asarray(start_idx, dtype=np.int64),
        "end_idx": np.asarray(end_idx, dtype=np.int64),
        "origin_hs": np.asarray(origin_hs, dtype=np.float32),
        "origin_time": np.asarray(origin_time, dtype="datetime64[ns]"),
        "X_by_station": x_by_station,
        "hs_by_station": hs_by_station,
        "source_frame": frame,
        "features": list(features),
        "n_features": len(features),
    }

WINDOW_META_KEYS = ("station_id", "start_idx", "end_idx", "origin_hs", "origin_time")

def chronological_split(samples, train_ratio=TRAIN_RATIO):
    times = pd.to_datetime(samples["origin_time"])
    cutoff = pd.Timestamp(pd.Series(times).quantile(train_ratio))

    def subset(mask):
        part = {key: samples[key][mask] for key in WINDOW_META_KEYS}
        part.update({
            "X_by_station": samples["X_by_station"],
            "hs_by_station": samples["hs_by_station"],
            "source_frame": samples["source_frame"],
            "features": samples["features"],
            "n_features": samples["n_features"],
            "split_cutoff": cutoff,
        })
        return part

    return subset(times <= cutoff), subset(times > cutoff), cutoff

def scale_samples(train, valid):
    # Fit directly on the original 2D train-period DataFrame, not on 3D windows.
    scaler = StandardScaler()
    train_rows = train["source_frame"][TIME_COL] <= train["split_cutoff"]
    scaler.fit(train["source_frame"].loc[train_rows, train["features"]])

    # train/valid share the same station matrices: transform each 2D matrix in place once.
    for Xv in train["X_by_station"].values():
        scaler.transform(Xv, copy=False)
    return dict(train), dict(valid), scaler


## 9. PyTorch Dataset

In [ ]:
# ============================================================
# 7. PYTORCH DATASET
# ============================================================
class WaveDataset(Dataset):
    def __init__(self, samples):
        self.X_by_station = samples["X_by_station"]
        self.hs_by_station = samples["hs_by_station"]
        self.station_id = samples["station_id"]
        self.start_idx = samples["start_idx"]
        self.end_idx = samples["end_idx"]
        self.origin_hs = samples["origin_hs"]
        self.lead_steps = np.asarray(LEAD_STEPS, dtype=np.int64)

    def __len__(self):
        return len(self.end_idx)

    def __getitem__(self, idx):
        sid = int(self.station_id[idx])
        start, end = int(self.start_idx[idx]), int(self.end_idx[idx])
        # Lazy views: only this batch's (289, n_features) windows are materialized.
        X = torch.from_numpy(self.X_by_station[sid][start:end + 1])
        y = torch.from_numpy(self.hs_by_station[sid][end + self.lead_steps])
        return X, y, torch.tensor(self.origin_hs[idx], dtype=torch.float32), torch.tensor(sid)

def make_loader(samples, batch_size=128, shuffle=False):
    ds = WaveDataset(samples)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )


## 10. iTransformer

In [ ]:
# ============================================================
# 8. iTransformer MODEL DEFINITION
# ============================================================
class ITransformer(nn.Module):
    def __init__(self, seq_len, n_features, pred_len, d_model=64, n_heads=2, e_layers=3, dropout=0.25, d_ff=None):
        super().__init__()
        self.seq_len = seq_len
        self.n_features = n_features
        self.pred_len = pred_len
        if d_ff is None:
            d_ff = d_model * 4

        self.value_embedding = nn.Linear(seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=e_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_features * d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, pred_len),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.value_embedding(x)
        x = self.encoder(x)
        x = self.norm(x)
        out = self.head(x)
        return out


## 11. Metrics & competition-aligned loss

In [ ]:
# ============================================================
# 9. METRICS & COMPETITION-ALIGNED LOSS
# ============================================================
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def lead_rmse(y_true, y_pred):
    return {f"rmse_{h}h": rmse(y_true[:, i], y_pred[:, i]) for i, h in enumerate(LEAD_HOURS)}

def competition_like_rmse(y_true, y_pred, origin_hs):
    mask = origin_hs >= 1.5
    if mask.sum() == 0:
        return np.nan, 0
    return rmse(y_true[mask], y_pred[mask]), int(mask.sum())

def competition_aligned_rmse(y_true, y_pred, origin_hs, threshold=1.5, high_weight=1.0, low_weight=0.3):
    diff_sq = (y_pred - y_true) ** 2
    weights = np.where(origin_hs >= threshold, high_weight, low_weight)[:, None]
    return float(np.sqrt(np.sum(weights * diff_sq) / np.sum(weights) + 1e-6))

def select_78h_separated_indices(times, station_ids, origin_hs, min_hours=78):
    selected = []
    times = pd.to_datetime(times)
    for sid in np.unique(station_ids):
        idx = np.where((station_ids == sid) & (origin_hs >= 1.5))[0]
        idx = idx[np.argsort(times[idx])]
        last_time = None
        for i in idx:
            t = times[i]
            if last_time is None or (t - last_time) >= pd.Timedelta(hours=min_hours):
                selected.append(i)
                last_time = t
    return np.asarray(selected, dtype=int)

def exact_competition_rmse(y_true, y_pred, samples):
    idx = select_78h_separated_indices(samples["origin_time"], samples["station_id"], samples["origin_hs"], min_hours=78)
    if len(idx) == 0:
        return np.nan, 0
    return rmse(y_true[idx], y_pred[idx]), len(idx)

# [EXP04 핵심] 대회 평가 지표 정렬 가중치 RMSE 손실함수
class CompetitionAlignedRMSELoss(nn.Module):
    def __init__(self, threshold=1.5, high_weight=1.0, low_weight=0.3):
        super().__init__()
        self.threshold = threshold
        self.high_weight = high_weight
        self.low_weight = low_weight

    def forward(self, pred, target, origin_hs):
        diff_sq = (pred - target) ** 2
        weights = torch.where(
            origin_hs >= self.threshold,
            torch.tensor(self.high_weight, device=pred.device),
            torch.tensor(self.low_weight, device=pred.device)
        ).unsqueeze(-1)
        weighted_mse = torch.sum(weights * diff_sq) / torch.sum(weights)
        return torch.sqrt(weighted_mse + 1e-6)


## 12. Training / evaluation

In [ ]:
# ============================================================
# 10. TRAIN & EVAL FUNCTIONS
# ============================================================
def evaluate_model(model, loader):
    model.eval()
    preds, ys, origins = [], [], []
    with torch.no_grad():
        for X, y, origin_hs, _ in loader:
            X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            pred = model(X)
            preds.append(pred.cpu().numpy())
            ys.append(y.cpu().numpy())
            origins.append(origin_hs.numpy())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    origin_hs = np.concatenate(origins)

    overall = rmse(y_true, y_pred)
    comp, comp_n = competition_like_rmse(y_true, y_pred, origin_hs)
    aligned = competition_aligned_rmse(y_true, y_pred, origin_hs)
    result = {
        "overall_rmse": overall,
        "comp_rmse": comp,
        "competition_aligned_rmse": aligned,
        "comp_valid_n": comp_n,
        **lead_rmse(y_true, y_pred),
    }
    return result, y_true, y_pred

def train_one_model(train_samples, valid_samples, params, max_epochs, patience, verbose=True):
    seed_everything(SEED)
    batch_size = params.get("batch_size", 128)
    train_loader = make_loader(train_samples, batch_size=batch_size, shuffle=True)
    valid_loader = make_loader(valid_samples, batch_size=batch_size, shuffle=False)

    model = ITransformer(
        seq_len=INPUT_LEN,
        n_features=train_samples["n_features"],
        pred_len=N_TARGETS,
        d_model=params["d_model"],
        n_heads=params["n_heads"],
        e_layers=params["e_layers"],
        dropout=params["dropout"],
        d_ff=params.get("d_ff", params["d_model"] * 4),
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    criterion = CompetitionAlignedRMSELoss(threshold=1.5, high_weight=1.0, low_weight=0.3)

    best_state = None
    best_score = np.inf
    wait = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        for X, y, origin_hs, _ in train_loader:
            X, y, origin_hs = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True), origin_hs.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred = model(X)
            loss = criterion(pred, y, origin_hs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        metrics, _, _ = evaluate_model(model, valid_loader)
        # Early stopping follows the same weighted RMSE as CompetitionAlignedRMSELoss.
        score = metrics["competition_aligned_rmse"]

        history.append({"epoch": epoch, "train_loss": float(np.mean(train_losses)), **metrics})
        if verbose:
            print(f"Epoch {epoch:02d} | train={np.mean(train_losses):.5f} | overall={metrics['overall_rmse']:.5f} | comp={metrics['comp_rmse']:.5f}")

        if score < best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    final_metrics, y_true, y_pred = evaluate_model(model, valid_loader)
    return model, final_metrics, pd.DataFrame(history), y_true, y_pred


In [ ]:
# ============================================================
# 13. PREPARE FINAL TRAIN / VALID SAMPLES
# ============================================================
best_samples = build_samples(df, BEST_FEATURES)
train_samples, valid_samples, split_cutoff = chronological_split(best_samples)
train_samples, valid_samples, best_scaler = scale_samples(train_samples, valid_samples)

print("split cutoff:", split_cutoff)
print("train windows:", len(train_samples["end_idx"]))
print("valid windows:", len(valid_samples["end_idx"]))
print("high-wave valid:", int((valid_samples["origin_hs"] >= 1.5).sum()))


In [ ]:
# ============================================================
# 14. FINAL RETRAIN — FIXED OPTUNA BEST PARAMS
# ============================================================
final_model, final_metrics, final_history, y_true, y_pred = train_one_model(
    train_samples,
    valid_samples,
    params=BEST_PARAMS,
    max_epochs=FINAL_EPOCHS,
    patience=FINAL_PATIENCE,
    verbose=True,
)

exact_comp_rmse, exact_comp_n = exact_competition_rmse(
    y_true, y_pred, valid_samples
)
final_metrics["exact_78h_comp_rmse"] = exact_comp_rmse
final_metrics["exact_78h_n"] = exact_comp_n

print("\n===== FINAL EXP04 METRICS =====")
print(pd.Series(final_metrics).to_frame("value"))

final_history.to_csv(EXP_DIR / "final_history.csv", index=False)
pd.DataFrame([final_metrics]).to_csv(EXP_DIR / "final_metrics.csv", index=False)


## 15. Save model / scaler / metadata

In [ ]:
# ============================================================
# 18. SAVE MODEL + SCALER + METADATA
# ============================================================
import joblib
torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "features": BEST_FEATURES,
        "feature_set_name": BEST_FEATURE_NAME,
        "input_len": INPUT_LEN,
        "lead_hours": LEAD_HOURS,
        "params": BEST_PARAMS,
        "station_to_id": STATION_TO_ID,
        "metrics": final_metrics,
    },
    EXP_DIR / "best_model.pt",
)
joblib.dump(best_scaler, EXP_DIR / "scaler.pkl")

metadata = {
    "data_path": str(DATA_PATH),
    "feature_set_name": BEST_FEATURE_NAME,
    "features": BEST_FEATURES,
    "input_len": INPUT_LEN,
    "lead_hours": LEAD_HOURS,
    "split_cutoff": str(split_cutoff),
    "train_n": int(len(train_samples["end_idx"])),
    "valid_n": int(len(valid_samples["end_idx"])),
    "best_params": BEST_PARAMS,
    "metrics": {k: float(v) if isinstance(v, (float, np.floating)) else int(v) if isinstance(v, (int, np.integer)) else v for k, v in final_metrics.items()},
}
with open(EXP_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("Saved EXP04 artifacts to:", EXP_DIR.resolve())


## 16. Test inference + submission

업로드된 `exp04.ipynb`의 NaN-safe test inference / submission 생성 코드를 그대로 사용합니다.
테스트 context 내부의 보간은 모두 예측 시점 이전 48시간 context 내부에서만 수행됩니다.


In [ ]:
# ============================================================
# 19. TEST INFERENCE + SUBMISSION CREATION (EXP04)
# NaN-safe inference
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch


# ============================================================
# 0. TRAIN-BASED FALLBACK STATISTICS
# ============================================================

# df = EXP04 학습에 사용한 최종 training dataframe
# 반드시 train/physics feature 생성 셀 실행 후 사용

BASE_FALLBACK_COLS = [
    "tp",
    "wspd",
    "gust",
    "wdir",
    "wvdir",
    "airt",
    "relh",
    "caph",
]


station_medians = (
    df
    .groupby("station")[BASE_FALLBACK_COLS]
    .median()
)

global_medians = (
    df[BASE_FALLBACK_COLS]
    .median()
)


# ------------------------------------------------------------
# HMAX / HS ratio
# ------------------------------------------------------------

ratio_df = df[
    df["hs"].notna()
    &
    df["hmax"].notna()
    &
    (df["hs"] > 0)
].copy()

ratio_df["hmax_hs_ratio"] = (
    ratio_df["hmax"] / ratio_df["hs"]
)

ratio_df = ratio_df[
    ratio_df["hmax_hs_ratio"].between(1.0, 3.0)
]

station_hmax_ratio = (
    ratio_df
    .groupby("station")["hmax_hs_ratio"]
    .median()
)

global_hmax_ratio = float(
    ratio_df["hmax_hs_ratio"].median()
)


def get_station_median(station, col):

    value = np.nan

    if (
        station in station_medians.index
        and
        col in station_medians.columns
    ):
        value = station_medians.loc[
            station,
            col
        ]

    if not np.isfinite(value):
        value = global_medians[col]

    return float(value)


def get_hmax_ratio(station):

    if station in station_hmax_ratio.index:

        value = station_hmax_ratio.loc[
            station
        ]

        if np.isfinite(value):
            return float(value)

    return global_hmax_ratio


# ============================================================
# 1. TEST FEATURE ENGINEERING
# ============================================================

def add_test_features_exp04(context):

    feature_frames = []

    fallback_log = []

    required_cols = [
        "case_id",
        "station",
        "step_minute",
        "hs",
        "tp",
        "hmax",
        "wvdir",
        "wspd",
        "gust",
        "wdir",
        "airt",
        "relh",
        "caph",
    ]

    missing = [
        c
        for c in required_cols
        if c not in context.columns
    ]

    if missing:
        raise ValueError(
            f"test_context missing columns: {missing}"
        )


    for case_id, group in context.groupby(
        "case_id",
        sort=False
    ):

        g = (
            group
            .sort_values("step_minute")
            .copy()
        )

        st = g["station"].iloc[0]


        # ====================================================
        # INPUT LENGTH CHECK
        # ====================================================

        if len(g) != INPUT_LEN:

            raise ValueError(
                f"{case_id}: expected {INPUT_LEN} rows, "
                f"got {len(g)}"
            )


        # ====================================================
        # 2. INVALID VALUES -> NaN
        # ====================================================

        for col in [
            "hs",
            "tp",
            "hmax",
        ]:

            g.loc[
                g[col] <= 0,
                col
            ] = np.nan


        g.loc[
            g["wspd"] < 0,
            "wspd"
        ] = np.nan


        g.loc[
            g["gust"] < 0,
            "gust"
        ] = np.nan


        g.loc[
            ~g["relh"].between(0, 100),
            "relh"
        ] = np.nan


        g.loc[
            ~g["caph"].between(950, 1050),
            "caph"
        ] = np.nan


        # ====================================================
        # 3. CASE-INTERNAL INTERPOLATION
        # ====================================================

        num_cols = [
            "hs",
            "tp",
            "hmax",
            "wspd",
            "gust",
            "airt",
            "relh",
            "caph",
        ]


        g[num_cols] = (
            g[num_cols]
            .interpolate(
                method="linear",
                limit_direction="both"
            )
            .ffill()
            .bfill()
        )


        # ====================================================
        # 4. DIRECTION
        # ====================================================

        for col in [
            "wdir",
            "wvdir",
        ]:

            g[col] = (
                g[col]
                .ffill()
                .bfill()
            )

            # case 전체 NaN일 경우
            if g[col].isna().any():

                n_missing = int(
                    g[col].isna().sum()
                )

                value = get_station_median(
                    st,
                    col
                )

                g[col] = (
                    g[col]
                    .fillna(value)
                )

                fallback_log.append({
                    "case_id": case_id,
                    "station": st,
                    "column": col,
                    "n": n_missing,
                    "method": "station_median",
                })


        g["wdir"] %= 360.0
        g["wvdir"] %= 360.0


        # ====================================================
        # 5. HS
        #
        # hs는 핵심 signal이라 전체 NaN이면 중단
        # ====================================================

        if g["hs"].isna().all():

            raise ValueError(
                f"{case_id}: hs is entirely missing"
            )


        # 일부만 남으면 nearest context fallback
        if g["hs"].isna().any():

            n_missing = int(
                g["hs"].isna().sum()
            )

            g["hs"] = (
                g["hs"]
                .ffill()
                .bfill()
            )

            fallback_log.append({
                "case_id": case_id,
                "station": st,
                "column": "hs",
                "n": n_missing,
                "method": "context_ffill_bfill",
            })


        # ====================================================
        # 6. HMAX
        #
        # case 전체 NaN이어도
        # hs * train station ratio로 복원
        # ====================================================

        ratio = get_hmax_ratio(st)


        hmax_missing = (
            g["hmax"].isna()
            &
            g["hs"].notna()
        )


        if hmax_missing.any():

            n_missing = int(
                hmax_missing.sum()
            )

            g.loc[
                hmax_missing,
                "hmax"
            ] = (
                g.loc[
                    hmax_missing,
                    "hs"
                ]
                *
                ratio
            )

            fallback_log.append({
                "case_id": case_id,
                "station": st,
                "column": "hmax",
                "n": n_missing,
                "method": "hs_station_ratio",
            })


        # ====================================================
        # 7. REMAINING BASE VARIABLES
        #
        # case 전체 NaN -> train station median
        # ====================================================

        fallback_cols = [
            "tp",
            "wspd",
            "gust",
            "airt",
            "relh",
            "caph",
        ]


        for col in fallback_cols:

            if g[col].isna().any():

                n_missing = int(
                    g[col].isna().sum()
                )

                value = get_station_median(
                    st,
                    col
                )

                g[col] = (
                    g[col]
                    .fillna(value)
                )

                fallback_log.append({
                    "case_id": case_id,
                    "station": st,
                    "column": col,
                    "n": n_missing,
                    "method": "station_median",
                })


        # ====================================================
        # 8. BASE SANITY
        # ====================================================

        g["hs"] = g["hs"].clip(
            lower=0.01
        )

        g["tp"] = g["tp"].clip(
            lower=0.01
        )

        g["wspd"] = g["wspd"].clip(
            lower=0.0
        )

        g["gust"] = g["gust"].clip(
            lower=0.0
        )


        g["hmax"] = np.maximum(
            g["hmax"],
            g["hs"]
        )


        g["gust"] = np.maximum(
            g["gust"],
            g["wspd"]
        )


        # ====================================================
        # 9. WIND / WAVE VECTORS
        # ====================================================

        wdir_rad = np.deg2rad(
            g["wdir"]
        )

        wvdir_rad = np.deg2rad(
            g["wvdir"]
        )


        g["u_wind"] = (
            g["wspd"]
            *
            np.sin(wdir_rad)
        )


        g["v_wind"] = (
            g["wspd"]
            *
            np.cos(wdir_rad)
        )


        g["u_wave"] = (
            g["hs"]
            *
            np.sin(wvdir_rad)
        )


        g["v_wave"] = (
            g["hs"]
            *
            np.cos(wvdir_rad)
        )


        # ====================================================
        # 10. WIND/WAVE ALIGNMENT
        # ====================================================

        diff = (
            (
                g["wdir"]
                -
                g["wvdir"]
                +
                180
            )
            %
            360
            -
            180
        )


        g["wind_wave_diff"] = (
            np.abs(diff)
        )


        g["wind_wave_alignment"] = (
            np.cos(
                np.deg2rad(diff)
            )
        )


        # ====================================================
        # 11. WAVE MOMENTUM
        # ====================================================

        g["hs_diff_1h"] = (
            g["hs"]
            -
            g["hs"].shift(6)
        )


        g["hs_diff_3h"] = (
            g["hs"]
            -
            g["hs"].shift(18)
        )


        g["hs_mean_6h"] = (
            g["hs"]
            .rolling(
                36,
                min_periods=1
            )
            .mean()
        )


        g["hs_mean_12h"] = (
            g["hs"]
            .rolling(
                72,
                min_periods=1
            )
            .mean()
        )


        g["hs_max_6h"] = (
            g["hs"]
            .rolling(
                36,
                min_periods=1
            )
            .max()
        )


        g["hs_max_12h"] = (
            g["hs"]
            .rolling(
                72,
                min_periods=1
            )
            .max()
        )


        # ====================================================
        # 12. WIND HISTORY
        # ====================================================

        g["wspd_mean_6h"] = (
            g["wspd"]
            .rolling(
                36,
                min_periods=1
            )
            .mean()
        )


        g["wspd_mean_12h"] = (
            g["wspd"]
            .rolling(
                72,
                min_periods=1
            )
            .mean()
        )


        g["gust_max_6h"] = (
            g["gust"]
            .rolling(
                36,
                min_periods=1
            )
            .max()
        )


        g["gust_max_12h"] = (
            g["gust"]
            .rolling(
                72,
                min_periods=1
            )
            .max()
        )


        g["gust_minus_wspd"] = (
            g["gust"]
            -
            g["wspd"]
        )


        # ====================================================
        # 13. PRESSURE TENDENCY
        # ====================================================

        g["caph_change_3h"] = (
            g["caph"]
            -
            g["caph"].shift(18)
        )


        g["caph_change_6h"] = (
            g["caph"]
            -
            g["caph"].shift(36)
        )


        g["caph_change_12h"] = (
            g["caph"]
            -
            g["caph"].shift(72)
        )


        # ====================================================
        # 14. WAVE DYNAMICS
        # ====================================================

        wl = (
            1.56
            *
            (g["tp"] ** 2)
        )


        g["wave_steepness"] = (
            g["hs"]
            /
            np.maximum(
                wl,
                1.0
            )
        )


        g["wave_energy"] = (
            g["hs"] ** 2
        )


        g["effective_wind_forcing"] = (
            (g["wspd"] ** 2)
            *
            g["wind_wave_alignment"]
        )


        # ====================================================
        # 15. INF -> NAN
        # ====================================================

        g = g.replace(
            [np.inf, -np.inf],
            np.nan
        )


        # ====================================================
        # 16. BEST FEATURES EXISTENCE CHECK
        # ====================================================

        missing_features = [
            c
            for c in BEST_FEATURES
            if c not in g.columns
        ]


        if missing_features:

            raise ValueError(
                f"{case_id}: missing features "
                f"{missing_features}"
            )


        # ====================================================
        # 17. DERIVED FEATURE INITIAL BOUNDARY
        #
        # shift(6), shift(18), shift(72) 등에서
        # 맨 앞부분에 생기는 NaN만 보정
        # ====================================================

        g[BEST_FEATURES] = (
            g[BEST_FEATURES]
            .bfill()
            .ffill()
        )


        # ====================================================
        # 18. FINAL FINITE CHECK
        # ====================================================

        X = g[
            BEST_FEATURES
        ].to_numpy(
            dtype=np.float32
        )


        if not np.isfinite(X).all():

            bad_cols = [
                col
                for col in BEST_FEATURES
                if not np.isfinite(
                    g[col]
                    .to_numpy(dtype=float)
                ).all()
            ]

            raise ValueError(
                f"{case_id}: non-finite features remain: "
                f"{bad_cols}"
            )


        feature_frames.append(g)


    # ========================================================
    # FALLBACK REPORT
    # ========================================================

    if fallback_log:

        fallback_df = pd.DataFrame(
            fallback_log
        )

        print(
            "\n===== INFERENCE FALLBACK SUMMARY ====="
        )

        display(
            fallback_df
            .groupby(
                ["column", "method"],
                as_index=False
            )
            .agg(
                n_cases=(
                    "case_id",
                    "nunique"
                ),
                n_values=(
                    "n",
                    "sum"
                ),
            )
        )


    return pd.concat(
        feature_frames,
        ignore_index=True
    )


# ============================================================
# 19. LOAD TEST
# ============================================================

test_context = pd.read_parquet(
    TEST_CONTEXT_PATH
)

test_index = pd.read_csv(
    TEST_INDEX_PATH
)


print(
    "test_context:",
    test_context.shape
)

print(
    "test_index:",
    test_index.shape
)


# ============================================================
# 20. ORIGINAL FULL-MISSING REPORT
# ============================================================

check_cols = [
    "hs",
    "tp",
    "hmax",
    "wvdir",
    "wspd",
    "gust",
    "wdir",
    "airt",
    "relh",
    "caph",
]


full_missing = (
    test_context
    .groupby("case_id")[check_cols]
    .apply(
        lambda x:
        x.isna().all()
    )
)


print(
    "\n===== ENTIRE CASE MISSING ====="
)

display(
    full_missing
    .sum()
    .sort_values(
        ascending=False
    )
)


# ============================================================
# 21. TEST FEATURE CREATION
# ============================================================

test_features = add_test_features_exp04(
    test_context
)


# ============================================================
# 22. WINDOWS
# ============================================================

case_order = (
    test_index["case_id"]
    .drop_duplicates()
    .tolist()
)


windows = []


for case_id in case_order:

    grp = (
        test_features[
            test_features["case_id"]
            ==
            case_id
        ]
        .sort_values(
            "step_minute"
        )
    )


    if len(grp) != INPUT_LEN:

        raise ValueError(
            f"{case_id}: "
            f"{len(grp)} rows "
            f"!= INPUT_LEN {INPUT_LEN}"
        )


    X = (
        grp[BEST_FEATURES]
        .to_numpy(
            dtype=np.float32
        )
    )


    if not np.isfinite(X).all():

        raise ValueError(
            f"{case_id}: "
            "NaN/Inf remains before scaling"
        )


    windows.append(X)


X_test_raw = np.stack(
    windows
)


print(
    "\nX_test_raw:",
    X_test_raw.shape
)


# ============================================================
# 23. SCALE
# ============================================================

X_test = (
    best_scaler
    .transform(
        X_test_raw.reshape(
            -1,
            len(BEST_FEATURES)
        )
    )
    .reshape(
        X_test_raw.shape
    )
    .astype(np.float32)
)


if not np.isfinite(
    X_test
).all():

    raise ValueError(
        "NaN/Inf created after scaling"
    )


# ============================================================
# 24. INFERENCE
# batch inference to reduce GPU memory
# ============================================================

final_model.eval()

pred_batches = []

INFER_BATCH_SIZE = 128


with torch.no_grad():

    for start in range(
        0,
        len(X_test),
        INFER_BATCH_SIZE
    ):

        xb = torch.from_numpy(
            X_test[
                start:
                start + INFER_BATCH_SIZE
            ]
        ).to(
            DEVICE
        )


        pred = (
            final_model(xb)
            .detach()
            .cpu()
            .numpy()
        )


        pred_batches.append(
            pred
        )


preds = np.concatenate(
    pred_batches,
    axis=0
)


if not np.isfinite(preds).all():

    raise ValueError(
        "Model prediction contains NaN/Inf"
    )


print(
    "prediction shape:",
    preds.shape
)


# ============================================================
# 25. BUILD SUBMISSION
# ============================================================

pred_rows = []


for case_id, row in zip(
    case_order,
    preds
):

    for lead_h, val in zip(
        LEAD_HOURS,
        row
    ):

        pred_rows.append({
            "case_id":
                case_id,

            "lead_h":
                lead_h,

            "hs_pred":
                float(val),
        })


pred_df = pd.DataFrame(
    pred_rows
)


submission = (
    test_index
    .merge(
        pred_df,
        on=[
            "case_id",
            "lead_h",
        ],
        how="left",
        validate="one_to_one"
    )
)


# ============================================================
# 26. FINAL PREDICTION SANITY
# ============================================================

if submission[
    "hs_pred"
].isna().any():

    bad = submission[
        submission["hs_pred"].isna()
    ]

    display(bad.head())

    raise ValueError(
        "Submission contains missing predictions"
    )


submission["hs_pred"] = (
    submission["hs_pred"]
    .clip(
        lower=0.0,
        upper=30.0
    )
)


# ============================================================
# 27. SAVE
# ============================================================

submission.to_csv(
    SUBMISSION_PATH,
    index=False,
    encoding="utf-8"
)


print(
    "\n"
    + "=" * 80
)

print(
    f"EXP04 SUBMISSION SAVED: "
    f"{SUBMISSION_PATH}"
)

print(
    "=" * 80
)

print(
    "rows:",
    len(submission)
)

print(
    "cases:",
    submission[
        "case_id"
    ].nunique()
)

print(
    "\nprediction stats:"
)

display(
    submission[
        "hs_pred"
    ]
    .describe()
)

display(
    submission.head(12)
)

In [ ]:
# ============================================================
# 17. FINAL REPRODUCTION SUMMARY / RUNTIME
# ============================================================
elapsed = time.time() - RUN_START

print("=" * 80)
print("FINAL REPRODUCTION COMPLETE")
print("=" * 80)
print("Feature set :", BEST_FEATURE_NAME, f"({len(BEST_FEATURES)} features)")
print("Best params :", BEST_PARAMS)
print("Split cutoff:", split_cutoff)
print("Model       :", EXP_DIR / "best_model.pt")
print("Scaler      :", EXP_DIR / "scaler.pkl")
print("Metadata    :", EXP_DIR / "metadata.json")
print("Submission  :", SUBMISSION_PATH)
print(f"Runtime     : {elapsed/60:.2f} minutes ({elapsed/3600:.3f} hours)")

if elapsed > 6 * 3600:
    print("[WARNING] Runtime exceeded the competition 6-hour limit.")
else:
    print("[OK] Runtime is within the 6-hour limit.")
